# Preprocessing scRNA-seq data

This tutorial notebook shows how a possible pipeline to prepare the expression data for scRepresenter, in order to achieve the best performance.

## Public PBMC3k dataset

Download the raw and processed PBMC3k datasets from Scanpy

In [1]:
import scanpy as sc

sc.settings.datasetdir = "../data"

raw_obj = sc.datasets.pbmc3k()
processed_obj = sc.datasets.pbmc3k_processed()

  0%|          | 0.00/5.58M [00:00<?, ?B/s]

  0%|          | 0.00/23.5M [00:00<?, ?B/s]

We only need the raw counts, however the processed dataset has cell type labels which we will use for the prediction task. Because of this, map the labels from the processed dataset into the raw dataset, and remove any cells with no corresponding cell type label.

In [2]:
raw_obj.obs['celltype'] = raw_obj.obs_names.map(lambda x: processed_obj.obs['louvain'].get(x, "unknown"))
raw_obj = raw_obj[raw_obj.obs['celltype'] != "unknown"]

Finally, remove duplicates and capitalize gene names.

In [3]:
raw_obj.var_names = raw_obj.var_names.str.capitalize()
raw_obj.var_names_make_unique()

## Preprocessing steps

First, filter out useless genes and cells.

In [4]:
sc.pp.filter_cells(raw_obj, min_genes=200)
sc.pp.filter_genes(raw_obj, min_cells=3)

Normalize the counts, and apply the log1p() function. For a better performance also filter out non-variable genes.

In [5]:
sc.pp.normalize_total(raw_obj, target_sum=1e4)
sc.pp.log1p(raw_obj)  
sc.pp.highly_variable_genes(raw_obj, n_top_genes=2000)
final_obj = raw_obj[:, raw_obj.var['highly_variable']].copy()

Finally, save the processed dataset as an h5ad file.

In [6]:
final_obj.write_h5ad("../data/pbmc3k_final.h5ad")